## Przykład: Regulator PID w piecu przemysłowym

Sterowanie temperaturą pieca przemysłowego z pełną analizą XAI

In [2]:
from omnixai.wrapper import ControlSystemWrapper
from omnixai.wrapper import SciPyWrapper
from omnixai.utils.engineering import PIDStabilityAnalyzer
from omnixai.explainers.engineering import (ShapEngineering, LIMEEngineering,)

from omnixai.visualization.dashboard import Dashboard
from omnixai.data.tabular import Tabular


In [3]:
import numpy as np
import matplotlib.pyplot as plt

import control as ct
from scipy import signal

from functools import lru_cache


In [4]:
class IndustrialFurnaceModel:
    def __init__(self):
        """Parametry fizyczne pieca przemysłowego dopasowane do stabilnych nastaw PID."""
        # Parametry termodynamiczne
        self.C = 1500.0      # Pojemność cieplna [J/K]
        self.R = 0.1         # Opór cieplny [K/W]
        self.tau1 = 12.0     # Dominująca stała czasowa [s]
        self.tau2 = 3.0      # Pomocnicza stała czasowa [s]
        self.theta = 10.0    # Opóźnienie transportowe [s]
        self.K = 0.1         # Wzmocnienie statyczne [K/kW]

        # Warunki otoczenia
        self.T_amb = 25.0    # Temperatura otoczenia [°C]

        # Ograniczenia fizyczne
        self.T_min = 0.0     # Minimalna temperatura [°C]
        self.T_max = 1200.0  # Maksymalna temperatura [°C]
        self.P_max = 50.0    # Maksymalna moc grzałki [kW]
        
    def create_transfer_function_control(self):
        num_delay, den_delay = ct.pade(self.theta, 2)
        delay_tf = ct.TransferFunction(num_delay, den_delay)

        num_plant = [self.K]
        den_plant = [self.tau1 * self.tau2, self.tau1 + self.tau2, 1]
        plant_tf = ct.TransferFunction(num_plant, den_plant)

        furnace_tf = ct.series(delay_tf, plant_tf)

        return furnace_tf

TEMP_ZAD=800

In [5]:
def ziegler_nichols_tuning(furnace):
    # Parametry obiektu
    L = furnace.theta
    T = furnace.tau1 + furnace.tau2
    K = furnace.K 

    Kp = 1.2 * T / (K * L)
    Ti = 2 * L
    Ki = Kp / Ti
    Td = 0.5 * L
    Kd = Kp * Td

    return {
        'Kp': Kp,
        'Ki': Ki,
        'Kd': Kd,
        'Ti': Ti,
        'Td': Td,
        'method': 'Ziegler-Nichols (Open-Loop)',
        'object_params': {'L': L, 'T': T, 'K': K}
    }

In [6]:
def create_pid_controller_control(Kp, Ki, Kd, N = 20):
    tau_f = Kd / N
    num = [Kd + Kp * tau_f, Kp + Ki * tau_f, Ki]
    den = [tau_f, 1.0, 0.0]

    return ct.TransferFunction(num, den)

def create_pid_controller_scipy(Kp, Ki, Kd, N = 20):
    tau_f = Kd / N
    num = [Kd + Kp * tau_f, Kp + Ki * tau_f, Ki]
    den = [tau_f, 1.0, 0.0]

    return signal.TransferFunction(num, den)

In [7]:
def simulate_closed_loop_system(furnace, Kp, Ki, Kd, setpoint=TEMP_ZAD, duration=1000.0):

    G = furnace.create_transfer_function_control()
    C = create_pid_controller_control(Kp, Ki, Kd)
    L = ct.series(C, G)  # Układ otwarty
    T_closed = ct.feedback(L, 1)  # Układ zamknięty
    wrapper = ControlSystemWrapper(T_closed)

    # Symulacja odpowiedzi skokowej
    t = np.linspace(0, duration, 5000)
    u = np.ones_like(t)

    # Odpowiedź układu (znormalizowana)
    y_norm = wrapper.predict(u, t=t)

    # Przeskalowanie do rzeczywistej temperatury
    y_temp = y_norm * (setpoint - furnace.T_amb) + furnace.T_amb

    # Obliczenie błędu
    error = setpoint - y_temp
    _, u_control = ct.forced_response(C, T=t, U=error)

    # Analiza metryk
    L = ct.series(C, G)  # Układ otwarty dla analizy
    analyzer = PIDStabilityAnalyzer(wrapper, L)
    metrics = analyzer.analyze()


    return {
        'time': t,
        'temperature': y_temp,
        'setpoint': setpoint,
        'control_signal': u_control,
        'error': error,
        'metrics': metrics,
        'wrapper': wrapper,
        'closed_loop_tf': T_closed
    }

In [8]:
def prepare_xai_training_data(furnace, n_samples=100, setpoint=TEMP_ZAD):

    print("PRZYGOTOWANIE DANYCH TRENINGOWYCH")


    # Parametry referencyjne (Ziegler-Nichols)
    zn_params = ziegler_nichols_tuning(furnace)
    Kp_ref, Ki_ref, Kd_ref = zn_params['Kp'], zn_params['Ki'], zn_params['Kd']

    print(f"\nParametry referencyjne (Ziegler-Nichols):")
    print(f"  Kp = {Kp_ref:.3f}")
    print(f"  Ki = {Ki_ref:.3f}")
    print(f"  Kd = {Kd_ref:.3f}")

    # Próbkowanie parametrów PID
    np.random.seed(42)

    Kp_samples = np.random.uniform(Kp_ref * 0.5, Kp_ref * 1.5, n_samples)
    Ki_samples = np.random.uniform(Ki_ref * 0.5, Ki_ref * 1.5, n_samples)
    Kd_samples = np.random.uniform(Kd_ref * 0.5, Kd_ref * 1.5, n_samples)

    X_train = np.column_stack([Kp_samples, Ki_samples, Kd_samples])

    print(f"\nWygenerowano {n_samples} konfiguracji PID")
    print(f"  Zakres Kp: [{Kp_samples.min():.2f}, {Kp_samples.max():.2f}]")
    print(f"  Zakres Ki: [{Ki_samples.min():.2f}, {Ki_samples.max():.2f}]")
    print(f"  Zakres Kd: [{Kd_samples.min():.2f}, {Kd_samples.max():.2f}]")

    print(f"\nSymulacja układów...")
    y_train = []

    for i, (Kp, Ki, Kd) in enumerate(X_train):
        try:
            result = simulate_closed_loop_system(
                furnace, Kp, Ki, Kd,
                setpoint=setpoint,
                duration=500.0)
            y_train.append(result['metrics']['mse'])
        except:
            y_train.append(1e6)  # Kara za niestabilny układ

        if (i + 1) % 20 == 0:
            print(f"  Postęp: {i+1}/{n_samples}")

    y_train = np.array(y_train)

    print(f"\nDane treningowe przygotowane")
    print(f"  MSE: min={y_train.min():.2f}, max={y_train.max():.2f}, mean={y_train.mean():.2f}")

    feature_names = ['Kp', 'Ki', 'Kd']

    return X_train, y_train, feature_names

In [9]:
def compare_pid_configurations(furnace, setpoint=TEMP_ZAD):
    # Parametry Zieglera-Nicholsa
    zn_params = ziegler_nichols_tuning(furnace)
    Kp_zn, Ki_zn, Kd_zn = zn_params['Kp'], zn_params['Ki'], zn_params['Kd']
    
    configurations = [
        {
            'name': 'Ziegler-Nichols',
            'Kp': Kp_zn,
            'Ki': Ki_zn,
            'Kd': Kd_zn,
            'description': 'Klasyczna metoda strojenia - kompromis'
        },
        {
            'name': 'Konserwatywny',
            'Kp': Kp_zn * 0.6,
            'Ki': Ki_zn * 0.6,
            'Kd': Kd_zn * 0.6,
            'description': 'Wolniejszy, ale bardziej stabilny'
        },
        {
            'name': 'Agresywny',
            'Kp': Kp_zn * 1.4,
            'Ki': Ki_zn * 1.4,
            'Kd': Kd_zn * 1.4,
            'description': 'Szybszy, ale z większym przeregulowaniem'
        },
        {
            'name': 'Tylko P',
            'Kp': Kp_zn,
            'Ki': 0.0,
            'Kd': 0.0,
            'description': 'Regulator proporcjonalny - błąd ustalony'
        },
        {
            'name': 'PI',
            'Kp': Kp_zn,
            'Ki': Ki_zn,
            'Kd': 0.0,
            'description': 'Bez członu różniczkującego - wolniejszy'
        }
    ]
    
    results = {}
    
    print("PORÓWNANIE KONFIGURACJI PID")
    
    for config in configurations:
        print(f"\n{config['name']}: {config['description']}")
        print(f"  Parametry: Kp={config['Kp']:.3f}, Ki={config['Ki']:.3f}, Kd={config['Kd']:.3f}")
        
        result = simulate_closed_loop_system(
            furnace,
            config['Kp'],
            config['Ki'],
            config['Kd'],
            setpoint=setpoint,
            duration=1000.0
        )
        
        metrics = result['metrics']
        print(f"  Metryki:")
        print(f"    MSE: {metrics['mse']:.2f}")
        print(f"    Czas ustalania: {metrics['settling_time_s']:.1f} s")
        print(f"    Przeregulowanie: {metrics['overshoot_percent']:.2f} %")
        if metrics.get('rise_time_s') is not None:
            print(f"    Czas narastania: {metrics['rise_time_s']:.1f} s")
        
        results[config['name']] = {
            'config': config,
            'simulation': result
        }
    return results

In [10]:
def create_comprehensive_plots(comparison_results, setpoint=TEMP_ZAD):

    # Wykres 1: Odpowiedzi temperaturowe
    print("\nTworzenie wykresu 1: Odpowiedzi temperaturowe...")
    fig1, ax1 = plt.subplots(figsize=(12, 6))
    for name, data in comparison_results.items():
        sim = data['simulation']
        ax1.plot(sim['time'], sim['temperature'], label=name, linewidth=2)

    ax1.axhline(y=setpoint, color='r', linestyle='--', label='Wartość zadana', linewidth=2)
    ax1.set_xlabel('Czas [s]', fontsize=12)
    ax1.set_ylabel('Temperatura [°C]', fontsize=12)
    ax1.set_title('Odpowiedzi temperaturowe dla różnych konfiguracji PID',
                  fontsize=14, fontweight='bold')
    ax1.legend(loc='best', fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim([0, 500])

    filename1 = 'chapter_6_plot_1_temperature_responses.png'
    plt.savefig(filename1, dpi=300, bbox_inches='tight')
    print(f"Wykres zapisany: {filename1}")
    plt.close(fig1)

    # Wykres 2: Sygnały sterujące
    print("\nTworzenie wykresu 2: Sygnały sterujące...")
    fig2, ax2 = plt.subplots(figsize=(10, 6))
    for name, data in comparison_results.items():
        sim = data['simulation']
        ax2.plot(sim['time'], sim['control_signal'], label=name, linewidth=2)

    ax2.set_xlabel('Czas [s]', fontsize=12)
    ax2.set_ylabel('Moc grzałki [kW]', fontsize=12)
    ax2.set_title('Sygnały sterujące', fontsize=14, fontweight='bold')
    ax2.legend(loc='best', fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim([0, 200])

    filename2 = 'chapter_6_plot_2_control_signals.png'
    plt.savefig(filename2, dpi=300, bbox_inches='tight')
    print(f"Wykres zapisany: {filename2}")
    plt.close(fig2)

    # Wykres 3: Porównanie metryk MSE
    print("\nTworzenie wykresu 3: Porównanie MSE...")
    fig3, ax3 = plt.subplots(figsize=(10, 6))
    names = list(comparison_results.keys())
    mse_values = [comparison_results[n]['simulation']['metrics']['mse'] for n in names]

    colors = plt.cm.viridis(np.linspace(0, 1, len(names)))
    bars = ax3.bar(range(len(names)), mse_values, color=colors, alpha=0.7)
    ax3.set_xlabel('Konfiguracja PID', fontsize=12)
    ax3.set_ylabel('MSE', fontsize=12)
    ax3.set_title('Porównanie MSE', fontsize=14, fontweight='bold')
    ax3.set_xticks(range(len(names)))
    ax3.set_xticklabels(names, rotation=45, ha='right', fontsize=12)
    ax3.grid(True, alpha=0.3, axis='y')

    filename3 = 'chapter_6_plot_3_mse_comparison.png'
    plt.savefig(filename3, dpi=300, bbox_inches='tight')
    print(f"Wykres zapisany: {filename3}")
    plt.close(fig3)

    # Wykres 4: Czas ustalania vs Przeregulowanie
    print("\nTworzenie wykresu 4: Czas ustalania vs Przeregulowanie...")
    fig4, ax4 = plt.subplots(figsize=(10, 6))
    settling_times = [comparison_results[n]['simulation']['metrics']['settling_time_s'] for n in names]
    overshoots = [comparison_results[n]['simulation']['metrics']['overshoot_percent'] for n in names]

    scatter = ax4.scatter(settling_times, overshoots, c=range(len(names)),
                         cmap='viridis', s=200, alpha=0.7, edgecolors='black', linewidth=2)

    for i, name in enumerate(names):
        if name == 'Agresywny':
            text=(8, -24)
        elif name == 'PI':
            text=(0, -24)
        elif name[0] == 'Z' or name[0] == 'K':
            text=(-18, -19)
        else:
            text=(0, 10)

        ax4.annotate(
        name,
        (settling_times[i], overshoots[i]),
        fontsize=12,
        ha='center',
        va='bottom',
        xytext=text,    
        textcoords='offset points'
    )


    ax4.set_xlabel('Czas ustalania [s]', fontsize=12)
    ax4.set_ylabel('Przeregulowanie [%]', fontsize=12)
    ax4.set_title('Czas ustalania vs Przeregulowanie',
                  fontsize=14, fontweight='bold')
    ax4.grid(True, alpha=0.3)

    filename4 = 'chapter_6_plot_4_settling_vs_overshoot.png'
    plt.savefig(filename4, dpi=300, bbox_inches='tight')
    print(f"Wykres zapisany: {filename4}")
    plt.close(fig4)

    # Wykres 5: Błędy regulacji
    print("\nTworzenie wykresu 5: Błędy regulacji...")
    fig5, ax5 = plt.subplots(figsize=(10, 6))
    for name, data in comparison_results.items():
        sim = data['simulation']
        ax5.plot(sim['time'], np.abs(sim['error']), label=name, linewidth=2)

    ax5.set_xlabel('Czas [s]', fontsize=12)
    ax5.set_ylabel('|Błąd| [°C]', fontsize=12)
    ax5.set_title('Wartość bezwzględna błędu regulacji',
                  fontsize=14, fontweight='bold')
    ax5.legend(loc='best', fontsize=10)
    ax5.grid(True, alpha=0.3)
    ax5.set_xlim([0, 500])
    ax5.set_yscale('log')

    filename5 = 'chapter_6_plot_5_error_regulation.png'
    plt.savefig(filename5, dpi=300, bbox_inches='tight')
    print(f"Wykres zapisany: {filename5}")
    plt.close(fig5)

    print("\n" + "-"*80)
    print("\nWygenerowane pliki:")
    print(f"- {filename1}")
    print(f"- {filename2}")
    print(f"- {filename3}")
    print(f"- {filename4}")
    print(f"- {filename5}")

In [11]:
def check_stability(furnace, Kp, Ki, Kd):
    try:
        G = furnace.create_transfer_function_control()
        C = create_pid_controller_control(Kp, Ki, Kd)
        L = ct.series(C, G)

        gm, pm, _, _ = ct.margin(L)

        gain_margin_db = 20 * np.log10(gm) if gm > 0 else -np.inf
        is_stable = (gain_margin_db > 0) and (pm > 0)

        print(f"Sprawdzanie stabilności: Kp={Kp:.3f}, Ki={Ki:.3f}, Kd={Kd:.3f} -> "
              f"GM={gain_margin_db:.1f}dB, PM={pm:.1f}° -> {'Stabilny' if is_stable else 'Niestały'}")

        return is_stable, gain_margin_db, pm

    except Exception:
        return False, -np.inf, -np.inf

In [12]:
def shap_analysis(furnace, X_train, y_train, test_configs):
    print("ANALIZA SHAP")
    _cache = {}

    def predict_mse(params_array):
        results = []
        for params in params_array:
            Kp = float(params[0])
            Ki = float(params[1])
            Kd = float(params[2])

            if Kp <= 0 or Ki <= 0 or Kd < 0:
                results.append(1e6)
                continue

            key = (round(Kp, 4), round(Ki, 4), round(Kd, 4))
            if key in _cache:
                results.append(_cache[key])
                continue

            try:
                result = simulate_closed_loop_system(
                    furnace, Kp, Ki, Kd,
                    setpoint=TEMP_ZAD,
                    duration=200.0
                )
                mse = result['metrics']['mse']
            except Exception:
                mse = 1e6

            _cache[key] = mse
            results.append(mse)

        return np.array(results)

    stable_mask = y_train < 1e5
    bg_stable = X_train[stable_mask][:20]
    if len(bg_stable) < 3:
        print("Za mało stabilnych próbek, używam X_train[:20]")
        bg_stable = X_train[:20]

    print(f"\nDane tła SHAP: {len(bg_stable)} próbek (numpy array)")
    print("Tworzenie explainera SHAP...")

    shap_explainer = ShapEngineering(
        predict_fn=predict_mse,
        background_data=bg_stable,
        feature_names=['Kp', 'Ki', 'Kd']
    )
    print("Explainer SHAP utworzony")
    config_names = list(test_configs.keys())
    all_params = np.array([
        [cfg['Kp'], cfg['Ki'], cfg['Kd']]
        for cfg in test_configs.values()
    ])  # shape (3, 3)

    print("\nMSE przed SHAP:")
    mse_values = {}
    for name, row in zip(config_names, all_params):
        mse = float(predict_mse(row.reshape(1, -1))[0])
        mse_values[name] = mse
        stab, gm, pm = check_stability(furnace, row[0], row[1], row[2])
        print(f"  {name}: MSE={mse:.4f}  GM={gm:.1f}dB  PM={pm:.1f}°  "
              f"{'OK' if stab else 'NOT STABLE'}")

    print(f"\nObliczanie SHAP dla {len(config_names)} instancji naraz...")
    explanation_all = shap_explainer.explain(all_params)
    print("Wartości SHAP obliczone")
    for i, name in enumerate(config_names):
        print(f"  Instancja {i} -> {name}")

    return {
        'explanation_all': explanation_all,
        'config_names': config_names,
        'mse_values': mse_values,
        'per_config': {
            name: {
                'config': test_configs[name],
                'mse': mse_values[name],
                'explanation': explanation_all,
            }
            for name in config_names
        }
    }

@lru_cache(maxsize=10000)
def cached_sim(Kp, Ki, Kd):
    result = simulate_closed_loop_system(...)
    return result['metrics']['mse']

def lime_analysis(furnace, X_train, y_train, test_configs):
    print("ANALIZA LIME")

    _cache = {}

    def predict_mse(params_array):
        results = []
        for params in params_array:
            Kp = float(params[0])
            Ki = float(params[1])
            Kd = float(params[2])

            if Kp <= 0 or Ki <= 0 or Kd < 0:
                results.append(1e6)
                continue

            key = (round(Kp, 4), round(Ki, 4), round(Kd, 4))
            if key in _cache:
                results.append(_cache[key])
                continue

            try:
                result = simulate_closed_loop_system(
                    furnace, Kp, Ki, Kd,
                    setpoint=TEMP_ZAD,
                    duration=200.0
                )
                mse = result['metrics']['mse']
            except Exception:
                mse = 1e6

            _cache[key] = mse
            results.append(mse)

        return np.array(results)

    stable_mask = y_train < 1e5
    bg_stable = X_train[stable_mask][:20]
    if len(bg_stable) < 3:
        bg_stable = X_train[:20]

    print(f"Dane tła LIME: {len(bg_stable)} próbek")
    print("Tworzenie explainera LIME...")

    try:
        lime_explainer = LIMEEngineering(
            predict_fn=predict_mse,
            background_data=bg_stable,
            mode="regression",
            feature_names=['Kp', 'Ki', 'Kd']
        )
    except (AssertionError, TypeError):
        lime_explainer = LIMEEngineering(
            predict_fn=predict_mse,
            background_data=bg_stable,
            mode="regression"
        )

    print("Explainer LIME utworzony")

    config_names = list(test_configs.keys())
    all_params = np.array([
        [cfg['Kp'], cfg['Ki'], cfg['Kd']]
        for cfg in test_configs.values()
    ])

    mse_values = {}
    for name, row in zip(config_names, all_params):
        mse = float(predict_mse(row.reshape(1, -1))[0])
        mse_values[name] = mse
        print(f"  {name}: MSE={mse:.4f}")

    print(f"\nObliczanie LIME dla {len(config_names)} instancji naraz...")
    explanation_all = lime_explainer.explain(
        all_params, num_features=3, num_samples=300
    )

    print("LIME obliczony")
    for i, name in enumerate(config_names):
        print(f"  Instancja {i} -> {name}")

    return {
        'explanation_all': explanation_all,
        'config_names': config_names,
        'mse_values': mse_values,
        'per_config': {
            name: {
                'config': test_configs[name],
                'mse': mse_values[name],
                'explanation': explanation_all,
            }
            for name in config_names
        }
    }


In [13]:
def comprehensive_analysis(comparison_results):
   
    print("Porównanie metryk jakości regulacji")
    print("-"*80)
    print(f"{'Konfiguracja':<20} {'MSE':<12} {'t_s [s]':<12} {'OS [%]':<12} {'t_r [s]':<12}")
    print("-"*80)

    for name, data in comparison_results.items():
        metrics = data['simulation']['metrics']
        rise_time = f"{metrics.get('rise_time_s', 'N/A'):.1f}" if metrics.get('rise_time_s') and metrics.get('rise_time_s') != 'N/A' else "N/A"
        settling_time = f"{metrics['settling_time_s']:.1f}" if isinstance(metrics['settling_time_s'], (int, float)) else str(metrics['settling_time_s'])
        print(f"{name:<20} {metrics['mse']:<12.2f} {settling_time:<12} "
              f"{metrics['overshoot_percent']:<12.2f} {rise_time:<12}")

    print("-"*80)
    print("PORÓWNANIE Z METODĄ ZIEGLERA-NICHOLSA")

    zn_metrics = comparison_results['Ziegler-Nichols']['simulation']['metrics']

    print("\nMetoda Zieglera-Nicholsa:")
    print(f"- MSE: {zn_metrics['mse']:.2f}")
    print(f"- Czas ustalania: {zn_metrics['settling_time_s']:.1f} s")
    print(f"- Przeregulowanie: {zn_metrics['overshoot_percent']:.2f} %")

    # Znajdź najlepszą konfigurację
    best_config = min(comparison_results.items(),
                     key=lambda x: x[1]['simulation']['metrics']['mse'])


    create_comprehensive_plots(comparison_results)

    return {
        'best_configuration': best_config[0],
        'zn_performance': zn_metrics,
        'recommendations': 'Użyj Zieglera-Nicholsa jako punktu startowego, następnie optymalizuj z XAI'
    }

In [14]:
def launch_dashboard(shap_results, lime_results):
    config_names = shap_results['config_names']

    all_params = np.array([
        [shap_results['per_config'][n]['config']['Kp'],
         shap_results['per_config'][n]['config']['Ki'],
         shap_results['per_config'][n]['config']['Kd']]
        for n in config_names
    ])
 
    instances = Tabular( all_params,feature_columns=['Kp', 'Ki', 'Kd'])

    print("DASHBOARD: http://127.0.0.1:8050")
    print("Mapa instancji:")
    for i, name in enumerate(config_names):
        print(f"  {i} -> {name}  MSE={shap_results['mse_values'][name]:.4f}")

    Dashboard(
        instances=instances,
        local_explanations={
            "SHAP - Wpływ Kp, Ki, Kd na MSE": shap_results['explanation_all'],
            "LIME - Lokalne gradienty MSE":      lime_results['explanation_all'],
        }
    ).show()

### Definicja pieca

In [15]:
print("Sterowanie temperaturą pieca przemysłowego")
# Inicjalizacja modelu
furnace = IndustrialFurnaceModel()

print("\nParametry fizyczne pieca:")
print(f"- Pojemność cieplna: C = {furnace.C} J/K")
print(f"- Opór cieplny: R = {furnace.R} K/W")
print(f"- Stała czasowa dominująca: τ₁ = {furnace.tau1} s")
print(f"- Stała czasowa pomocnicza: τ₂ = {furnace.tau2} s")
print(f"- Opóźnienie transportowe: θ = {furnace.theta} s")
print(f"- Wzmocnienie statyczne: K = {furnace.K} K/kW")


# Strojenie Zieglera-Nicholsa
print("\nStrojenie metodą Zieglera-Nicholsa:")
zn_params = ziegler_nichols_tuning(furnace)
print(f"\nParametry PID (Ziegler-Nichols):")
print(f"- Kp = {zn_params['Kp']:.3f}")
print(f"- Ki = {zn_params['Ki']:.3f}")
print(f"- Kd = {zn_params['Kd']:.3f}")
print(f"- Ti = {zn_params['Ti']:.3f} s")
print(f"- Td = {zn_params['Td']:.3f} s")


Sterowanie temperaturą pieca przemysłowego

Parametry fizyczne pieca:
- Pojemność cieplna: C = 1500.0 J/K
- Opór cieplny: R = 0.1 K/W
- Stała czasowa dominująca: τ₁ = 12.0 s
- Stała czasowa pomocnicza: τ₂ = 3.0 s
- Opóźnienie transportowe: θ = 10.0 s
- Wzmocnienie statyczne: K = 0.1 K/kW

Strojenie metodą Zieglera-Nicholsa:

Parametry PID (Ziegler-Nichols):
- Kp = 18.000
- Ki = 0.900
- Kd = 90.000
- Ti = 20.000 s
- Td = 5.000 s


In [16]:
# Porównanie konfiguracji
comparison_results = compare_pid_configurations(furnace, setpoint=TEMP_ZAD)

PORÓWNANIE KONFIGURACJI PID

Ziegler-Nichols: Klasyczna metoda strojenia - kompromis
  Parametry: Kp=18.000, Ki=0.900, Kd=90.000
  Metryki:
    MSE: 0.60
    Czas ustalania: 29.8 s
    Przeregulowanie: 3.18 %

Konserwatywny: Wolniejszy, ale bardziej stabilny
  Parametry: Kp=10.800, Ki=0.540, Kd=54.000
  Metryki:
    MSE: 0.48
    Czas ustalania: 30.0 s
    Przeregulowanie: 1.12 %

Agresywny: Szybszy, ale z większym przeregulowaniem
  Parametry: Kp=25.200, Ki=1.260, Kd=126.000
  Metryki:
    MSE: 1.04
    Czas ustalania: 29.2 s
    Przeregulowanie: 5.12 %

Tylko P: Regulator proporcjonalny - błąd ustalony
  Parametry: Kp=18.000, Ki=0.000, Kd=0.000
  Metryki:
    MSE: 0.50
    Czas ustalania: 30.0 s
    Przeregulowanie: 1.97 %

PI: Bez członu różniczkującego - wolniejszy
  Parametry: Kp=18.000, Ki=0.900, Kd=0.000
  Metryki:
    MSE: 0.58
    Czas ustalania: 29.4 s
    Przeregulowanie: 8.75 %


### Przygotowanie danych treningowych

In [17]:
# Przygotowanie danych treningowych
X_train, y_train, feature_names = prepare_xai_training_data(furnace, n_samples=100, setpoint=TEMP_ZAD)

# Funkcja pomocnicza do zaokrąglania parametrów
def round_params(params, digits=2):
    return {k: round(v, digits) for k, v in params.items()}

# Konfiguracje testowe dla XAI z zaokrągleniem
test_configs = {
    'Ziegler-Nichols': round_params({
        'Kp': zn_params['Kp'],
        'Ki': zn_params['Ki'],
        'Kd': zn_params['Kd']
    }),
    'Konserwatywny': round_params({
        'Kp': zn_params['Kp'] * 0.6,
        'Ki': zn_params['Ki'] * 0.6,
        'Kd': zn_params['Kd'] * 0.6
    }),
    'Agresywny': round_params({
        'Kp': zn_params['Kp'] * 1.4,
        'Ki': zn_params['Ki'] * 1.4,
        'Kd': zn_params['Kd'] * 1.4
    })
}

# Wyświetlenie dla weryfikacji
for cfg, params in test_configs.items():
    print(cfg, params)

PRZYGOTOWANIE DANYCH TRENINGOWYCH

Parametry referencyjne (Ziegler-Nichols):
  Kp = 18.000
  Ki = 0.900
  Kd = 90.000

Wygenerowano 100 konfiguracji PID
  Zakres Kp: [9.10, 26.76]
  Zakres Ki: [0.46, 1.34]
  Zakres Kd: [45.46, 134.10]

Symulacja układów...
  Postęp: 20/100
  Postęp: 40/100
  Postęp: 60/100
  Postęp: 80/100
  Postęp: 100/100

Dane treningowe przygotowane
  MSE: min=0.47, max=0.95, mean=0.63
Ziegler-Nichols {'Kp': 18.0, 'Ki': 0.9, 'Kd': 90.0}
Konserwatywny {'Kp': 10.8, 'Ki': 0.54, 'Kd': 54.0}
Agresywny {'Kp': 25.2, 'Ki': 1.26, 'Kd': 126.0}


### Analiza XAI

In [18]:
# Analiza SHAP
shap_results = shap_analysis(furnace, X_train, y_train, test_configs)

ANALIZA SHAP

Dane tła SHAP: 20 próbek (numpy array)
Tworzenie explainera SHAP...
Explainer SHAP utworzony

MSE przed SHAP:
Sprawdzanie stabilności: Kp=18.000, Ki=0.900, Kd=90.000 → GM=0.0dB, PM=0.3° → Stabilny
  Ziegler-Nichols: MSE=0.6033  GM=0.0dB  PM=0.3°  OK
Sprawdzanie stabilności: Kp=10.800, Ki=0.540, Kd=54.000 → GM=5.0dB, PM=71.9° → Stabilny
  Konserwatywny: MSE=0.4773  GM=5.0dB  PM=71.9°  OK
Sprawdzanie stabilności: Kp=25.200, Ki=1.260, Kd=126.000 → GM=-3.1dB, PM=-61.2° → Niestały
  Agresywny: MSE=1.0378  GM=-3.1dB  PM=-61.2°  NOT STABLE

Obliczanie SHAP dla 3 instancji naraz...


  0%|          | 0/3 [00:00<?, ?it/s]

Wartości SHAP obliczone
  Instancja 0 → Ziegler-Nichols
  Instancja 1 → Konserwatywny
  Instancja 2 → Agresywny


In [19]:
# Analiza LIME
lime_results = lime_analysis(furnace, X_train, y_train, test_configs)

ANALIZA LIME
Dane tła LIME: 20 próbek
Tworzenie explainera LIME...
Explainer LIME utworzony
  Ziegler-Nichols: MSE=0.6033
  Konserwatywny: MSE=0.4773
  Agresywny: MSE=1.0378

Obliczanie LIME dla 3 instancji naraz...
LIME obliczony
  Instancja 0 → Ziegler-Nichols
  Instancja 1 → Konserwatywny
  Instancja 2 → Agresywny


In [20]:
analysis_report = comprehensive_analysis(comparison_results)

Porównanie metryk jakości regulacji
--------------------------------------------------------------------------------
Konfiguracja         MSE          t_s [s]      OS [%]       t_r [s]     
--------------------------------------------------------------------------------
Ziegler-Nichols      0.60         29.8         3.18         N/A         
Konserwatywny        0.48         30.0         1.12         N/A         
Agresywny            1.04         29.2         5.12         N/A         
Tylko P              0.50         30.0         1.97         N/A         
PI                   0.58         29.4         8.75         N/A         
--------------------------------------------------------------------------------
PORÓWNANIE Z METODĄ ZIEGLERA-NICHOLSA

Metoda Zieglera-Nicholsa:
- MSE: 0.60
- Czas ustalania: 29.8 s
- Przeregulowanie: 3.18 %

Tworzenie wykresu 1: Odpowiedzi temperaturowe...
Wykres zapisany: chapter_6_plot_1_temperature_responses.png

Tworzenie wykresu 2: Sygnały sterujące...
Wy

In [22]:
launch_dashboard(shap_results, lime_results)


DASHBOARD: http://127.0.0.1:8050
Mapa instancji:
  0 → Ziegler-Nichols  MSE=0.6033
  1 → Konserwatywny  MSE=0.4773
  2 → Agresywny  MSE=1.0378
